In [1]:
# HAIKUTHON - rerank SPEED/ACCURACY trade-off test (team conditions)
# Stage 1 SigLIP2-L zero-shot -> top-4, Stage 2 Qwen3-VL-Reranker-2B yes/no score.
# Tests VLM image size, prompt length, int8 weights (needed for the 5GB zip), K (2/3/4) and
# "rerank only ambiguous queries" (fixed stage-1 margin threshold), then prints one table:
# score vs estimated L4 time for the real test (16,000 images, 8,000 queries).
# Kaggle Notebook: GPU T4 x2 (only cuda:0 used), Internet On. Outputs in /kaggle/working/ (re-run resumes).
# Split = team rule: np.random.RandomState(42).permutation(len(ds)), first 500 -> val.

In [2]:
# %% [1] install (pip skips anything already satisfied). Qwen3-VL needs transformers >= 4.57.
!pip install -q "transformers>=4.57.0"

In [3]:
# %% [2] settings
STAGE1 = "google/siglip2-large-patch16-384"
TAG = "siglip2L"                       # same prefix as zeroshot_eval_siglip2L -> its saved embeddings are reused
RERANKER = "Qwen/Qwen3-VL-Reranker-2B"
OUT, DEVICE = "/kaggle/working", "cuda:0"
KMAX = 4                               # candidates scored per query (K=2/3 are evaluated from the same scores)
CONFIGS = [                            # each config = one pass over 500 x KMAX pairs (~4 min on T4)
    dict(name="r448_full",      res=448, prompt="full",  int8=False),   # = previous test
    dict(name="r320_full",      res=320, prompt="full",  int8=False),   # 100 instead of 196 image tokens
    dict(name="r320_nosys",     res=320, prompt="nosys", int8=False),   # drop the system message
    dict(name="r320_full_int8", res=320, prompt="full",  int8=True),    # int8-stored weights (zip < 5GB)
]
BATCH = 16                             # pairs per forward; lower if CUDA OOM
ALPHAS = [1, 2, 4, None]               # final = z(stage1) + alpha*z(rerank); None = rerank score only
T4_TO_L4 = 1.8                         # ponytail: rough fp16 FLOPs ratio, verify with one real submission
SERVER_CPUS = 4                        # PNG decoding on the server is assumed to use all 4 vCPUs
LIMIT_MIN = 13.5                       # leave a safety margin under the 15-minute limit
N_Q, N_IMG = 8000, 16000

import os, json, time, re
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
os.makedirs(OUT, exist_ok=True)

In [4]:
# %% [3] dataset + team split
from datasets import load_dataset
ds = load_dataset("KimWooJin/26-haikuthon-train", split="train")
CAP = next(c for c in ["prompts", "prompt", "text", "caption"] if c in ds.column_names)
print(ds.column_names, "| caption column:", CAP)
idx = np.random.RandomState(42).permutation(len(ds))
val_idx = sorted(idx[:500].tolist())
json.dump(val_idx, open(f"{OUT}/val_idx.json", "w"))
FIRST10 = val_idx[:10]
print("first 10 val_idx:", FIRST10)
VORIG = [ds[i]["image"].convert("RGB") for i in val_idx]       # 500 original images (for both stages)
RAW = [ds[i][CAP].strip() for i in val_idx]                     # original casing for the VLM
LOW = [t.lower() for t in RAW]                                  # SigLIP2 was trained on lowercased text

README.md:   0%|          | 0.00/4.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

images/0001.png: reconstructing file:   0%|          |  0.00B / 1.20MB            

images/0000.png: reconstructing file:   0%|          |  0.00B /  977kB            

images/0014.png: reconstructing file:   0%|          |  0.00B /  834kB            

... (7,996 outputs left out)

images/3999.png: downloading bytes:           |  0.00B            

metadata.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

['image', 'id', 'prompts'] | caption column: prompts
first 10 val_idx: [8, 12, 14, 17, 23, 26, 32, 33, 44, 45]


In [5]:
# %% [4] stage 1: SigLIP2-L zero-shot (official: max_length=64 padding), reuse saved embeddings if present
from transformers import AutoModel, AutoProcessor
proc1 = AutoProcessor.from_pretrained(STAGE1)
s1 = AutoModel.from_pretrained(STAGE1, attn_implementation="sdpa").to(DEVICE).half().eval()
s1_gb = sum(p.numel() for p in s1.parameters()) * 2 / 1e9
lens = np.array([len(x) for x in proc1.tokenizer(LOW)["input_ids"]])
trunc = float((lens > 64).mean())
img_path, spd_path = f"{OUT}/{TAG}_val_img_emb.npy", f"{OUT}/{TAG}_speed.json"
with torch.no_grad():
    if os.path.exists(img_path) and os.path.exists(spd_path):
        I = torch.from_numpy(np.load(img_path)); sp = json.load(open(spd_path)); print("reused", img_path)
    else:
        s1.vision_model(pixel_values=proc1(images=VORIG[:1], return_tensors="pt")["pixel_values"].to(DEVICE).half())  # warm-up
        cpu_t = gpu_t = 0.0; E = []
        for s in range(0, 500, 32):
            t0 = time.time(); pv = proc1(images=VORIG[s:s + 32], return_tensors="pt")["pixel_values"]; cpu_t += time.time() - t0
            torch.cuda.synchronize(DEVICE); t0 = time.time()
            E.append(F.normalize(s1.vision_model(pixel_values=pv.to(DEVICE).half()).pooler_output.float(), dim=-1).cpu())
            torch.cuda.synchronize(DEVICE); gpu_t += time.time() - t0
        I = torch.cat(E); np.save(img_path, I.numpy())
        sp = {"gpu_img_per_s": 500 / gpu_t, "e2e_img_per_s": 500 / (cpu_t + gpu_t)}; json.dump(sp, open(spd_path, "w"))
    torch.cuda.synchronize(DEVICE); t0 = time.time(); E = []
    for s in range(0, 500, 64):
        ids = proc1(text=LOW[s:s + 64], padding="max_length", max_length=64, truncation=True, return_tensors="pt")["input_ids"].to(DEVICE)
        E.append(F.normalize(s1.text_model(input_ids=ids).pooler_output.float(), dim=-1).cpu())
    torch.cuda.synchronize(DEVICE); txt_s_per = (time.time() - t0) / 500
    T = torch.cat(E); np.save(f"{OUT}/{TAG}_val_txt_emb.npy", T.numpy())
del s1; torch.cuda.empty_cache()

cpu_s_per = max(1 / sp["e2e_img_per_s"] - 1 / sp["gpu_img_per_s"], 0)       # decode+preprocess, one core
s1_min = (N_IMG / (sp["gpu_img_per_s"] * T4_TO_L4) + N_IMG * cpu_s_per / SERVER_CPUS + N_Q * txt_s_per / T4_TO_L4) / 60
sim = T @ I.T
rank1 = ((sim > sim.diag()[:, None]).sum(1) + 1).numpy()
topv, topi = sim.topk(KMAX, dim=1)
margin = (topv[:, 0] - topv[:, 1]).numpy()                                   # small margin = ambiguous query
print(f"stage-1 est. on L4: {s1_min:.1f} min | correct in top-2/3/4: "
      + " / ".join(f"{(rank1 <= k).mean():.3f}" for k in (2, 3, 4)) + f" | >64 tokens: {trunc:.1%}")

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.53GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

stage-1 est. on L4: 4.0 min | correct in top-2/3/4: 0.950 / 0.994 / 0.996 | >64 tokens: 2.0%


In [6]:
# %% [5] reranker: prompt formats, int8 round-trip, scoring with resume
from transformers import AutoProcessor as AP, Qwen3VLForConditionalGeneration
JUDGE_SYS = 'Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".'
INSTR = "<Instruct>: Retrieve images that match the user's description exactly.<Query>:"
def conv(prompt, text, img):   # format follows Qwen3-VL-Reranker's scripts/qwen3_vl_reranker.py (approximated)
    user = {"role": "user", "content": [{"type": "text", "text": INSTR}, {"type": "text", "text": text},
                                        {"type": "text", "text": "\n<Document>:"}, {"type": "image", "image": img}]}
    return [user] if prompt == "nosys" else [{"role": "system", "content": [{"type": "text", "text": JUDGE_SYS}]}, user]

def int8_roundtrip_(m):
    """Per-output-row symmetric int8 on every large 2-D weight (what we would store in the zip),
    dequantised back to fp16 in place so accuracy can be measured. Returns the stored size in GB."""
    q = o = 0
    with torch.no_grad():
        for n, p in m.named_parameters():
            if p.dim() == 2 and p.numel() >= 65536:
                w = p.data.float(); s = w.abs().amax(1, keepdim=True).clamp(min=1e-8) / 127
                p.data.copy_(((w / s).round().clamp(-127, 127) * s).to(p.dtype)); q += p.numel() + s.numel() * 2
                del w
            else:
                o += p.numel() * 2
    torch.cuda.empty_cache()
    return (q + o) / 1e9

proc2 = AP.from_pretrained(RERANKER); proc2.tokenizer.padding_side = "left"   # last position = answer slot
vocab = proc2.tokenizer.get_vocab()
YES, NO = [vocab[w] for w in ["yes", "Yes"] if w in vocab], [vocab[w] for w in ["no", "No"] if w in vocab]
state = {"m": None, "int8": None}
def get_model(int8):
    if state["m"] is None or state["int8"] != int8:
        state["m"] = None; torch.cuda.empty_cache()
        m = Qwen3VLForConditionalGeneration.from_pretrained(RERANKER, dtype=torch.float16, attn_implementation="sdpa").to(DEVICE).eval()
        state.update(m=m, int8=int8, gb=int8_roundtrip_(m) if int8 else sum(p.numel() for p in m.parameters()) * 2 / 1e9)
    return state["m"]

def score_pairs(m, prompt, imgs, pairs, bs):
    W = m.lm_head.weight[YES + NO].float()
    out = torch.zeros(len(pairs))
    with torch.no_grad():
        for s in range(0, len(pairs), bs):
            ch = pairs[s:s + bs]
            inp = proc2.apply_chat_template([conv(prompt, RAW[a], imgs[int(topi[a, b])]) for a, b in ch], tokenize=True,
                                            add_generation_prompt=True, return_dict=True, padding=True, return_tensors="pt").to(DEVICE)
            lg = m.model(**inp).last_hidden_state[:, -1].float() @ W.T
            out[s:s + len(ch)] = (torch.logsumexp(lg[:, :len(YES)], 1) - torch.logsumexp(lg[:, len(YES):], 1)).cpu()
    return out

RES = {}
pairs = [(a, b) for a in range(500) for b in range(KMAX)]
for c in sorted(CONFIGS, key=lambda c: c["int8"]):            # fp16 configs first, then quantise once
    f = f"{OUT}/rerank_{c['name']}.pt"
    if os.path.exists(f):
        RES[c["name"]] = torch.load(f); print("reused", f); continue
    m = get_model(c["int8"])
    imgs = [im.resize((c["res"], c["res"]), Image.BICUBIC) for im in VORIG]
    score_pairs(m, c["prompt"], imgs, pairs[:BATCH], BATCH)                # warm-up
    torch.cuda.synchronize(DEVICE); t0 = time.time()
    R = score_pairs(m, c["prompt"], imgs, pairs, BATCH).view(500, KMAX)
    torch.cuda.synchronize(DEVICE); pps = len(pairs) / (time.time() - t0)
    if torch.isnan(R).any(): print(f"WARNING {c['name']}: NaN scores (fp16 overflow?)")
    RES[c["name"]] = {"R": R, "pps": pps, "gb": state["gb"]}; torch.save(RES[c["name"]], f)
    print(f"{c['name']}: {pps:.1f} pairs/s (T4, batch {BATCH}), reranker weights {state['gb']:.2f} GB")

preprocessor_config.json:   0%|          | 0.00/628 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.29k [00:00<?, ?B/s]

reranker.jinja:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.45k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (15 outputs left out)

r320_nosys: 15.8 pairs/s (T4, batch 16), reranker weights 4.26 GB


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

r320_full_int8: 14.0 pairs/s (T4, batch 16), reranker weights 2.13 GB


In [7]:
# %% [6] evaluate every combination -> table (score vs estimated L4 time)
def metrics(r):
    r = np.asarray(r, float); r1 = (r == 1).astype(float); nd = np.where(r <= 5, 1 / np.log2(r + 1), 0)
    return r1.mean(), nd.mean(), ((r1 + nd) / 2).mean()
def zrow(x): return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)
def ranks_after(R, K, gate, alpha):
    f = R[:, :K].clone() if alpha is None else zrow(topv[:, :K]) + alpha * zrow(R[:, :K])
    order = f.argsort(1, descending=True); new = rank1.copy()
    for a in np.where(gate & (rank1 <= K))[0]:
        new[a] = (topi[a, :K][order[a]] == a).nonzero()[0, 0].item() + 1
    return new

# thresholds are ABSOLUTE stage-1 margins picked on val (a fixed constant at test time = per-query, allowed)
THR = [("all", np.inf)] + [(f"margin<{np.quantile(margin, qq):.4f}", float(np.quantile(margin, qq))) for qq in (0.75, 0.5, 0.25)]
r1_, nd_, sc_ = metrics(rank1)
rows = [dict(config="stage-1 only", K=0, rerank="-", alpha="-", frac=0.0, R1=r1_, NDCG5=nd_, score=sc_, est_min=s1_min, gb=s1_gb)]
for name, d in RES.items():
    for K in (2, 3, 4):
        for tname, t in THR:
            gate = margin < t; best = None
            for al in ALPHAS:
                m3 = metrics(ranks_after(d["R"], K, gate, al))
                if best is None or m3[2] > best[1][2]: best = (al, m3)
            est = s1_min + N_Q * K * gate.mean() / (d["pps"] * T4_TO_L4) / 60
            rows.append(dict(config=name, K=K, rerank=tname, alpha="only" if best[0] is None else best[0], frac=gate.mean(),
                             R1=best[1][0], NDCG5=best[1][1], score=best[1][2], est_min=est, gb=s1_gb + d["gb"]))
tab = pd.DataFrame(rows).sort_values("score", ascending=False)
tab.to_csv(f"{OUT}/rerank_speed_table.csv", index=False)
pd.set_option("display.width", 200)
fmt = {"frac": "{:.0%}".format, "R1": "{:.4f}".format, "NDCG5": "{:.4f}".format, "score": "{:.4f}".format,
       "est_min": "{:.1f}".format, "gb": "{:.2f}".format}
fits = tab[(tab.est_min <= LIMIT_MIN) & (tab.gb <= 5.0)]
print(f"=== fits in {LIMIT_MIN} min AND 5 GB (top 10) ===");  print(fits.head(10).to_string(index=False, formatters=fmt))
print("\n=== all combinations (top 15 by score) ==="); print(tab.head(15).to_string(index=False, formatters=fmt))
print("\nnote: alpha and thresholds chosen on these same 500 val queries (slightly optimistic); "
      f"time = T4 measurement / {T4_TO_L4} + decode estimate, not measured on L4; frac = share of queries reranked")

=== fits in 13.5 min AND 5 GB (top 10) ===
        config  K        rerank alpha frac     R1  NDCG5  score est_min   gb
r320_full_int8  3 margin<0.0446     2  50% 0.8540 0.9408 0.8974    11.9 3.90
r320_full_int8  2 margin<0.1125     1  75% 0.8480 0.9352 0.8916    11.9 3.90
r320_full_int8  2 margin<0.0446     1  50% 0.8480 0.9352 0.8916     9.3 3.90
r320_full_int8  4 margin<0.0130     1  25% 0.8240 0.9278 0.8759     9.3 3.90
r320_full_int8  2 margin<0.0130     1  25% 0.8240 0.9264 0.8752     6.6 3.90
r320_full_int8  3 margin<0.0130     2  25% 0.8200 0.9267 0.8734     7.9 3.90
  stage-1 only  0             -     -   0% 0.7860 0.9123 0.8492     4.0 1.76

=== all combinations (top 15 by score) ===
    config  K        rerank alpha frac     R1  NDCG5  score est_min   gb
 r320_full  4 margin<0.1125     4  75% 0.8720 0.9470 0.9095    19.5 6.02
 r320_full  4           all     4 100% 0.8720 0.9470 0.9095    24.7 6.02
 r320_full  4 margin<0.0446     4  50% 0.8720 0.9464 0.9092    14.3 6.02
 r320

In [8]:
# %% [7] team-share line for the best combination that fits
b = fits.iloc[0] if len(fits) else tab.iloc[0]
if not len(fits): print("nothing fits -> showing the best overall; needs more speed-up")
name = "siglip2-large-patch16-384" + ("" if b.config == "stage-1 only" else
       f" + Qwen3-VL-Reranker-2B [{b.config}, K={b.K}, {b.rerank}, alpha={b.alpha}]")
print("\n[팀 공유용]")
print(f"{name} / {FIRST10} / R@1 {b.R1:.4f} / NDCG@5 {b.NDCG5:.4f} / 최종 {b.score:.4f} / "
      f"16,000장 {b.est_min:.1f}분 (L4 추정, 재정렬 포함) / {b.gb:.2f} GB / 잘림 {trunc:.1%} (SigLIP >64 tok, VLM 0%)")


[팀 공유용]
siglip2-large-patch16-384 + Qwen3-VL-Reranker-2B [r320_full_int8, K=3, margin<0.0446, alpha=2] / [8, 12, 14, 17, 23, 26, 32, 33, 44, 45] / R@1 0.8540 / NDCG@5 0.9408 / 최종 0.8974 / 16,000장 11.9분 (L4 추정, 재정렬 포함) / 3.90 GB / 잘림 2.0% (SigLIP >64 tok, VLM 0%)


In [9]:
# %% [8] (optional) throughput vs batch size for the current model - pick BATCH for model.py
imgs = [im.resize((320, 320), Image.BICUBIC) for im in VORIG]
for bs in (8, 16, 32):
    try:
        m = state["m"]; score_pairs(m, "full", imgs, pairs[:bs], bs)
        torch.cuda.synchronize(DEVICE); t0 = time.time(); score_pairs(m, "full", imgs, pairs[:256], bs)
        torch.cuda.synchronize(DEVICE); print(f"batch {bs}: {256 / (time.time() - t0):.1f} pairs/s (T4, 320px)")
    except torch.cuda.OutOfMemoryError:
        print(f"batch {bs}: CUDA OOM"); torch.cuda.empty_cache(); break

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

batch 8: 14.0 pairs/s (T4, 320px)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

batch 16: 13.6 pairs/s (T4, 320px)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch 32: 13.9 pairs/s (T4, 320px)
